<a href="https://colab.research.google.com/github/jygheo/Audio-Sheet-Music-Retrieval/blob/main/usage_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y poppler-utils
!pip install pdf2image ultralytics

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
from huggingface_hub import hf_hub_download
import torch
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch.nn as nn
from torchvision.models import swin_v2_t, Swin_V2_T_Weights
import torch.nn.functional as F
import gradio as gr
from pdf2image import convert_from_path
import torchaudio
import torchaudio.transforms as T_audio
import torchvision.transforms as T
import soundfile as sf
import requests
import os

In [ ]:
class SheetMusicSwin(nn.Module):
    def __init__(self, out_channels=512):
        super(SheetMusicSwin, self).__init__()

        self.backbone = swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)

        in_features = self.backbone.head.in_features

        self.backbone.head = nn.Identity()

        self.projection_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, out_channels)
        )

    def forward(self, x):
        """
        x: Tensor of shape [batch_size, 3, H, W]
        """
        #[batch_size, 1024]
        features = self.backbone(x)

        # [batch_size, 512])
        projected = self.projection_head(features)

        out = F.normalize(projected, p=2, dim=1)

        return out

In [ ]:
class SpectrogramSwin(nn.Module):
    def __init__(self, out_channels=512):
        super(SpectrogramSwin, self).__init__()

        self.backbone = swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)

        original_conv = self.backbone.features[0][0]

        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding
        )


        new_conv.weight.data = original_conv.weight.data.mean(dim=1, keepdim=True)
        if original_conv.bias is not None:
            new_conv.bias.data = original_conv.bias.data

        self.backbone.features[0][0] = new_conv

        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.projection_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, out_channels)
        )

    def forward(self, x):
        """
        x: Spectrogram Tensor of shape [batch_size, 1, 92, num_frames]
        """

        H, W = x.shape[2], x.shape[3]
        pad_h = (32 - H % 32) % 32
        pad_w = (32 - W % 32) % 32

        x = F.pad(x, (0, pad_w, 0, pad_h))

        features = self.backbone(x)

        projected = self.projection_head(features)

        out = F.normalize(projected, p=2, dim=1)

        return out

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

segmenter_path = hf_hub_download(
    repo_id="hyg444/audio-sheet-music-retrieval",
    filename="piano_line_segmenter.pt"
)

audio_path = hf_hub_download(
    repo_id="hyg444/audio-sheet-music-retrieval",
    filename="audio_encoder.pth"
)

vision_path = hf_hub_download(
    repo_id="hyg444/audio-sheet-music-retrieval",
    filename="vision_encoder.pth"
)

filterbank_path = hf_hub_download(
    repo_id="hyg444/audio-sheet-music-retrieval",
    filename="madmom_filterbank.pt"
)

segmenter_model = torch.hub.load('ultralytics/yolov5', 'custom', path=segmenter_path)

vision_encoder = SheetMusicSwin().to(device)
audio_encoder = SpectrogramSwin().to(device)

vision_encoder.load_state_dict(torch.load(vision_path, map_location=device))
audio_encoder.load_state_dict(torch.load(audio_path, map_location=device))

vision_encoder.eval()
audio_encoder.eval()

filterbank = torch.load(filterbank_path, map_location=device)


piano_line_segmenter.pt:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

audio_encoder.pth:   0%|          | 0.00/113M [00:00<?, ?B/s]

vision_encoder.pth:   0%|          | 0.00/113M [00:00<?, ?B/s]

madmom_filterbank.pt:   0%|          | 0.00/440k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['urllib3>=2.6.0 ; python_version > "3.8"'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WA

YOLOv5 🚀 2026-4-23 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth


100%|██████████| 109M/109M [00:00<00:00, 212MB/s]


In [ ]:
def download_file(url, filename):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    with open(filename, 'wb') as f:
        f.write(response.content)

download_file(
    "https://s9.imslp.org/files/imglnks/usimg/e/e4/IMSLP06163-Ravel_-_Pavane_pour_une_infante_d%C3%A9funte_(Piano).pdf",
    "default_score.pdf"
)

download_file(
    "https://d19bhbirxx14bg.cloudfront.net/ravel-pavane-griner.mp3",
    "default_audio.mp3"
)

In [ ]:
def letterbox_image(pil_img, target_w, target_h, pad_color=(255, 255, 255)):

    orig_w, orig_h = pil_img.size
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)

    resized = pil_img.resize((new_w, new_h), Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", (target_w, target_h), color=pad_color)

    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    canvas.paste(resized, (paste_x, paste_y))

    return canvas

def slice_piano_sheet_music_from_image(pil_img, target_w=416, target_h=128, margin=40):
    """
    Detects staves using a YOLO model, crops them with a safety margin,
    and standardizes the output size.
    """
    results = segmenter_model(pil_img, size=640)

    predictions = results.pandas().xyxy[0]

    predictions = predictions.sort_values('ymin')

    page_width, page_height = pil_img.size
    crops_pil = []

    for _, row in predictions.iterrows():
        y1, y2 = int(row['ymin']), int(row['ymax'])

        crop_top = max(0, y1 - margin)
        crop_bottom = min(page_height, y2 + margin)

        #lol prob change this
        crop_left = 0
        crop_right = page_width

        crop_img = pil_img.crop((crop_left, crop_top, crop_right, crop_bottom))

        final_canvas = letterbox_image(crop_img, target_w, target_h)
        crops_pil.append(final_canvas)

    return crops_pil

def process_pdf(pdf_path):
    if not pdf_path:
        return []

    pages = convert_from_path(pdf_path, dpi=200)

    all_crops = []
    for page_num, page_img in enumerate(pages):
        page_crops = slice_piano_sheet_music_from_image(page_img)
        all_crops.extend(page_crops)

    return all_crops

In [ ]:
def slice_audio_snippet(audio_filepath, start_sec, duration_sec):
    info = sf.info(audio_filepath)
    sr = info.samplerate

    start_frame = int(start_sec * sr)
    num_frames = int(duration_sec * sr)

    waveform, sample_rate = torchaudio.load(
        audio_filepath,
        frame_offset=start_frame,
        num_frames=num_frames
    )

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    return waveform, sample_rate


In [ ]:
class MSMDSpectrogramPipeline(torch.nn.Module):
    def __init__(self, filterbank):
        super().__init__()
        self.fps = 20
        self.n_fft = 2048
        self.hop_length = 1102 # 22050 / 20

        self.register_buffer('filterbank', filterbank)

        self.spectrogram = torchaudio.transforms.Spectrogram(
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            power=1.0
        )

    def forward(self, waveform):
        mag_spec = self.spectrogram(waveform)

        filtered_spec = torch.einsum('bft,fm->bmt', mag_spec, self.filterbank)

        normalized_spec = (filtered_spec - filtered_spec.mean()) / (filtered_spec.std() + 1e-6)

        return normalized_spec

In [ ]:
audio_pipeline = MSMDSpectrogramPipeline(filterbank=filterbank).to(device)
audio_pipeline.eval()
transform = T.Compose([
    T.ToTensor()])

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Audio-Sheet Music Retrieval
        Upload a PDF of piano sheet music and a full audio recording, or use the default examples. Use the sliders to select a 4-8 second snippet of the audio to find exactly where it occurs in the score!
        """
    )

    pdf_crops_state = gr.State([])
    pdf_embeds_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            #  default files
            pdf_input = gr.File(label="1. Upload Piano Score (PDF)", file_types=[".pdf"], value="default_score.pdf")
            audio_input = gr.Audio(type="filepath", label="2. Upload Audio Recording", value="default_audio.mp3")

            with gr.Group():
                gr.Markdown("### Select your Audio Snippet")

                start_time = gr.Slider(
                    minimum=0,
                    maximum=10,
                    step=0.1,
                    label="Snippet Start Time (seconds)",
                    interactive=False
                )

                clip_length = gr.Slider(
                    minimum=4.0,
                    maximum=8.0,
                    step=0.5,
                    value=5.0,
                    label="Snippet Duration (seconds)",
                    interactive=True
                )

            search_btn = gr.Button("Find Location in Score", variant="primary")

        with gr.Column(scale=1):
            match_output = gr.Image(label="Matched Sheet Music Line")
            confidence_out = gr.Textbox(label="Confidence Score")


    def update_slider_max(audio_path):
        """When audio is uploaded, find its length and unlock the Start Time slider."""
        if not audio_path:
            return gr.update(interactive=False, value=0)

        info = sf.info(audio_path)
        total_duration = info.frames / info.samplerate

        return gr.update(
            maximum=total_duration,
            interactive=True,
            info=f"Total Audio Length: {total_duration:.1f}s"
        )

    def process_new_pdf(pdf_file):
        if not pdf_file:
            return [], None

        crops_pil = process_pdf(pdf_file)

        if not crops_pil:
            return [], None

        # Pre-compute all vision embeddings and cache them!
        vision_tensors = torch.stack([transform(img) for img in crops_pil]).to(device)
        with torch.no_grad():
            vision_embeds = vision_encoder(vision_tensors)

        return crops_pil, vision_embeds


    def process_search(crops_pil, vision_embeds, audio_filepath, start_sec, duration_sec):
        if not crops_pil or vision_embeds is None:
            return None, "Error: Wait for the PDF to finish processing."
        if not audio_filepath:
            return None, "Error: Missing Audio."

        waveform, sr = slice_audio_snippet(audio_filepath, start_sec, duration_sec)
        waveform = waveform.to(device)

        if sr != 22050:
            resampler = T_audio.Resample(orig_freq=sr, new_freq=22050).to(device)
            waveform = resampler(waveform)

        spectrogram = audio_pipeline(waveform).unsqueeze(0)

        with torch.no_grad():
            audio_embed = audio_encoder(spectrogram)

        similarities = torch.matmul(audio_embed, vision_embeds.T).squeeze(0)
        best_idx = torch.argmax(similarities).item()
        confidence = similarities[best_idx].item()

        return crops_pil[best_idx], f"{confidence:.3f}"


    pdf_input.change(fn=process_new_pdf, inputs=pdf_input, outputs=[pdf_crops_state, pdf_embeds_state])
    audio_input.change(fn=update_slider_max, inputs=audio_input, outputs=start_time)

    search_btn.click(
        fn=process_search,
        inputs=[pdf_crops_state, pdf_embeds_state, audio_input, start_time, clip_length],
        outputs=[match_output, confidence_out]
    )

    demo.load(fn=process_new_pdf, inputs=pdf_input, outputs=[pdf_crops_state, pdf_embeds_state])
    demo.load(fn=update_slider_max, inputs=audio_input, outputs=start_time)

current_dir = os.path.abspath(".")
demo.launch(share=True, debug=True, allowed_paths=[current_dir])

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b7adda51add33d8419.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


New PDF detected. Slicing staves and computing visual embeddings...
Extracting pages from /tmp/gradio/349a4354f6bcf4507863d7b91b0a9e6379d13406f672743ef230b656d6ce3469/default_score.pdf...
Slicing page 1...


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Slicing page 2...


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


...

✅ PDF Processing Complete! Ready for search.
New PDF detected. Slicing staves and computing visual embeddings...
Extracting pages from /tmp/gradio/349a4354f6bcf4507863d7b91b0a9e6379d13406f672743ef230b656d6ce3469/default_score.pdf...
Slicing page 1...
Slicing page 2...
Slicing page 3...


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Slicing page 4...
✅ PDF Processing Complete! Ready for search.


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b7adda51add33d8419.gradio.live
